In [ ]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files


# ============================================================
# 2. UPLOAD FILE promotions.csv
# ============================================================

uploaded = files.upload()


# ============================================================
# 3. TỰ ĐỘNG TÌM FILE PROMOTION
# ============================================================

promotion_file = None

for filename in uploaded.keys():
    name = filename.lower()

    if "promotion" in name and name.endswith(".csv"):
        promotion_file = filename
        break


if promotion_file is None:
    raise FileNotFoundError(
        "Không tìm thấy file promotions.csv"
    )


print("File được sử dụng:", promotion_file)


# ============================================================
# 4. ĐỌC DỮ LIỆU
# ============================================================

promotions = pd.read_csv(
    promotion_file,
    low_memory=False
)

print("\nSố dòng ban đầu:", len(promotions))
print("Số cột ban đầu:", len(promotions.columns))

display(promotions.head())


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
#
# Ví dụ:
# "Promo ID" -> promo_id
# " DISCOUNT VALUE " -> discount_value
# ============================================================

promotions.columns = (
    promotions.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)


# ============================================================
# 6. ĐỒNG BỘ TÊN CỘT app_category
#
# File gốc:
# applicable_category
#
# Lược đồ:
# app_category
# ============================================================

if (
    "applicable_category" in promotions.columns
    and
    "app_category" not in promotions.columns
):

    promotions = promotions.rename(
        columns={
            "applicable_category": "app_category"
        }
    )


# ============================================================
# 7. CHUẨN HÓA CÁC GIÁ TRỊ ĐƯỢC XEM LÀ MISSING
# ============================================================

missing_values = [
    "",
    " ",
    "NULL",
    "null",
    "None",
    "none",
    "N/A",
    "n/a",
    "NA",
    "na",
    "-",
    "--",
    "?"
]


text_columns = promotions.select_dtypes(
    include=["object", "string"]
).columns


for col in text_columns:

    promotions[col] = (
        promotions[col]
        .astype("string")
        .str.strip()
    )

    promotions[col] = (
        promotions[col]
        .replace(
            missing_values,
            pd.NA
        )
    )


# ============================================================
# 8. CHUẨN HÓA KIỂU DỮ LIỆU SỐ
# ============================================================

numeric_columns = [
    "discount_value",
    "min_order_value"
]


for col in numeric_columns:

    if col in promotions.columns:

        promotions[col] = pd.to_numeric(
            promotions[col],
            errors="coerce"
        )


# ============================================================
# 9. CHUẨN HÓA stackable_flag
#
# 0 = không stack
# 1 = có thể stack
# ============================================================

if "stackable_flag" in promotions.columns:

    promotions["stackable_flag"] = pd.to_numeric(
        promotions["stackable_flag"],
        errors="coerce"
    ).astype("Int64")


# ============================================================
# 10. CHUẨN HÓA NGÀY THÁNG
# ============================================================

date_columns = [
    "start_date",
    "end_date"
]


for col in date_columns:

    if col in promotions.columns:

        promotions[col] = pd.to_datetime(
            promotions[col],
            errors="coerce"
        )


# ============================================================
# 11. KIỂM TRA MISSING TRƯỚC KHI XỬ LÝ
# ============================================================

print(
    "\n===== MISSING VALUE TRƯỚC CLEANING ====="
)

print(
    promotions.isna().sum()
)


# ============================================================
# 12. XỬ LÝ app_category
#
# Nếu bị thiếu:
# -> xem là promotion áp dụng cho tất cả category
# -> điền "All"
#
# Giữ mỗi promo_id chỉ 1 dòng.
# ============================================================

if "app_category" in promotions.columns:

    promotions["app_category"] = (
        promotions["app_category"]
        .fillna("All")
    )


# ============================================================
# 13. XỬ LÝ MISSING CHO DỮ LIỆU SỐ
#
# Kỹ thuật:
# Group-wise Median Imputation
#
# Ưu tiên median theo promo_type.
# Nếu vẫn thiếu -> median toàn bảng.
# ============================================================

for col in [
    "discount_value",
    "min_order_value"
]:

    if (
        col in promotions.columns
        and
        promotions[col].isna().any()
    ):

        if "promo_type" in promotions.columns:

            group_median = (
                promotions
                .groupby("promo_type")[col]
                .transform("median")
            )

            promotions[col] = (
                promotions[col]
                .fillna(group_median)
            )

        # Fallback: median toàn bộ cột
        promotions[col] = (
            promotions[col]
            .fillna(
                promotions[col].median()
            )
        )


# ============================================================
# 14. XỬ LÝ stackable_flag BỊ THIẾU
#
# Dùng Mode Imputation
# ============================================================

if (
    "stackable_flag" in promotions.columns
    and
    promotions["stackable_flag"].isna().any()
):

    mode_value = (
        promotions["stackable_flag"]
        .mode(dropna=True)
    )

    if not mode_value.empty:

        promotions["stackable_flag"] = (
            promotions["stackable_flag"]
            .fillna(
                mode_value.iloc[0]
            )
        )


# ============================================================
# 15. XỬ LÝ CÁC BIẾN CATEGORICAL BỊ THIẾU
#
# Dùng Mode Imputation
# ============================================================

categorical_columns = [
    "promo_name",
    "promo_type",
    "promo_channel"
]


for col in categorical_columns:

    if (
        col in promotions.columns
        and
        promotions[col].isna().any()
    ):

        mode_value = (
            promotions[col]
            .mode(dropna=True)
        )

        if not mode_value.empty:

            promotions[col] = (
                promotions[col]
                .fillna(
                    mode_value.iloc[0]
                )
            )


# ============================================================
# 16. KIỂM TRA promo_id BỊ TRỐNG
#
# promo_id là identifier nên KHÔNG được dùng mean/mode
# để tự suy đoán.
# ============================================================

if "promo_id" in promotions.columns:

    missing_promo_id = (
        promotions["promo_id"]
        .isna()
        .sum()
    )

    print(
        "\nSố promo_id bị thiếu:",
        missing_promo_id
    )


# ============================================================
# 17. KIỂM TRA promo_id BỊ TRÙNG
# ============================================================

duplicate_count = (
    promotions["promo_id"]
    .duplicated()
    .sum()
)


print(
    "Số promo_id bị trùng:",
    duplicate_count
)


if duplicate_count > 0:

    print(
        "\nCác promo_id bị trùng:"
    )

    display(
        promotions[
            promotions["promo_id"]
            .duplicated(keep=False)
        ]
        .sort_values("promo_id")
    )


# ============================================================
# 18. NẾU DÒNG BỊ TRÙNG HOÀN TOÀN
# THÌ XÓA DÒNG TRÙNG
#
# Không tự xóa chỉ dựa trên promo_id.
# ============================================================

before = len(promotions)

promotions = (
    promotions
    .drop_duplicates()
    .reset_index(drop=True)
)

after = len(promotions)


print(
    "\nSố dòng trùng hoàn toàn đã xóa:",
    before - after
)


# ============================================================
# 19. KIỂM TRA NGÀY THÁNG
#
# start_date không được lớn hơn end_date
# ============================================================

if (
    "start_date" in promotions.columns
    and
    "end_date" in promotions.columns
):

    invalid_date = (
        promotions["start_date"].notna()
        &
        promotions["end_date"].notna()
        &
        (
            promotions["start_date"]
            >
            promotions["end_date"]
        )
    )

    print(
        "Số dòng start_date > end_date:",
        invalid_date.sum()
    )

    if invalid_date.sum() > 0:

        display(
            promotions.loc[
                invalid_date,
                [
                    "promo_id",
                    "promo_name",
                    "start_date",
                    "end_date"
                ]
            ]
        )


# ============================================================
# 20. KIỂM TRA GIÁ TRỊ SỐ BẤT HỢP LÝ
# ============================================================

if "discount_value" in promotions.columns:

    invalid_discount = (
        promotions["discount_value"] < 0
    )

    print(
        "Số discount_value < 0:",
        invalid_discount.sum()
    )


if "min_order_value" in promotions.columns:

    invalid_min_order = (
        promotions["min_order_value"] < 0
    )

    print(
        "Số min_order_value < 0:",
        invalid_min_order.sum()
    )


# ============================================================
# 21. KIỂM TRA stackable_flag
#
# Chỉ cho phép 0 hoặc 1
# ============================================================

if "stackable_flag" in promotions.columns:

    invalid_stackable = (
        ~promotions["stackable_flag"]
        .isin([0, 1])
        &
        promotions["stackable_flag"]
        .notna()
    )

    print(
        "Số stackable_flag không phải 0/1:",
        invalid_stackable.sum()
    )


# ============================================================
# 22. KIỂM TRA MISSING SAU KHI CLEAN
# ============================================================

print(
    "\n===== MISSING VALUE SAU CLEANING ====="
)

print(
    promotions.isna().sum()
)


# ============================================================
# 23. KIỂM TRA app_category
# ============================================================

print(
    "\n===== PHÂN BỐ app_category ====="
)

print(
    promotions["app_category"]
    .value_counts(dropna=False)
)


# ============================================================
# 24. HIỂN THỊ DỮ LIỆU CUỐI
# ============================================================

print(
    "\n===== PROMOTION SAU CHUẨN HÓA ====="
)

display(
    promotions.head(20)
)


print(
    "\nSố dòng cuối cùng:",
    len(promotions)
)

print(
    "Số promo_id khác nhau:",
    promotions["promo_id"].nunique()
)


# ============================================================
# 25. XUẤT FILE promotion_new.csv
# ============================================================

promotions.to_csv(
    "promotion_new.csv",
    index=False,
    date_format="%Y-%m-%d"
)


print(
    "\nĐã tạo file promotion_new.csv"
)


# ============================================================
# 26. DOWNLOAD FILE
# ============================================================

files.download(
    "promotion_new.csv"
)